# 🧠📍 Smart Tagging

Metrica's **automatic** tactical annotation of the tactical video: phases of play and set
pieces, each one a time interval with a code and a set of tags.

This notebook covers loading it, what is in it, and the two things that will bite you if you
treat it as ground truth.

---

## Loading

The file is a **SportsCode XML**, the standard format in football performance analysis, so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/sportscode/) reads it directly.

`games.asset_paths()` builds the path, so nothing here spells a filename out and
`MTAH_DATA_DIR` keeps working if you store the bundle elsewhere.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from kloppy import sportscode

import games

path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
print('%d instances' % len(events))
events.head()

638 instances


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,NaN,NaN


In [2]:
events_df = pd.DataFrame(events)
events_df.sort_values(by='timestamp', inplace=True)

In [3]:
events_df

,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
103,104,1,0 days 00:07:22.760000,0 days 00:08:33.600000,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
403,404,1,0 days 00:07:23.400000,0 days 00:07:34.760000,LONG BALL,Atletic Club,NaN,1st Half,None,NaN,NaN
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN
542,543,1,0 days 00:07:28.160000,0 days 00:08:00.840000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,1st Half,None,NaN,NaN
318,319,1,0 days 00:07:28.160000,0 days 00:08:00.840000,PROGRESSION,Atletic Club,NaN,1st Half,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
396,397,1,0 days 02:05:10.080000,0 days 02:05:26.440000,LONG BALL,Barcelona,NaN,2nd Half,None,NaN,NaN
448,449,1,0 days 02:05:37.280000,0 days 02:06:12.280000,SET PIECES,N/A,Throw-in,2nd Half,None,NaN,NaN
500,501,1,0 days 02:05:46.640000,0 days 02:06:09.440000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,2nd Half,None,NaN,NaN
276,277,1,0 days 02:05:46.640000,0 days 02:06:09.440000,PROGRESSION,Atletic Club,NaN,2nd Half,None,NaN,NaN


In [4]:
# type(events_df['end_timestamp'][0])
events_df['end_frame'] = (events_df['end_timestamp'].dt.total_seconds()*25).round().astype(int)
events_df['start_frame'] = (events_df['timestamp'].dt.total_seconds()*25).round().astype(int)

In [5]:
events_df.sort_values(by='start_frame')

,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry,end_frame,start_frame
103,104,1,0 days 00:07:22.760000,0 days 00:08:33.600000,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN,12840,11069
403,404,1,0 days 00:07:23.400000,0 days 00:07:34.760000,LONG BALL,Atletic Club,NaN,1st Half,None,NaN,NaN,11369,11085
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN,11222,11097
542,543,1,0 days 00:07:28.160000,0 days 00:08:00.840000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,1st Half,None,NaN,NaN,12021,11204
318,319,1,0 days 00:07:28.160000,0 days 00:08:00.840000,PROGRESSION,Atletic Club,NaN,1st Half,None,NaN,NaN,12021,11204
...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,397,1,0 days 02:05:10.080000,0 days 02:05:26.440000,LONG BALL,Barcelona,NaN,2nd Half,None,NaN,NaN,188161,187752
448,449,1,0 days 02:05:37.280000,0 days 02:06:12.280000,SET PIECES,N/A,Throw-in,2nd Half,None,NaN,NaN,189307,188432
500,501,1,0 days 02:05:46.640000,0 days 02:06:09.440000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,2nd Half,None,NaN,NaN,189236,188666
276,277,1,0 days 02:05:46.640000,0 days 02:06:09.440000,PROGRESSION,Atletic Club,NaN,2nd Half,None,NaN,NaN,189236,188666


In [6]:
attacking_codes = [
    "BALL IN FINAL THIRD",
    "BALL IN THE BOX",
    "PROGRESSION",
    "CREATING CHANCES",
    "LONG BALL",
    "ATTACKING TRANSITION",
    "BUILD UP",
    "GOALS"
]

defending_codes = [
    "DEFENDING IN MIDDLE THIRD",
    "DEFENDING IN DEFENSIVE THIRD",
    "DEFENSIVE TRANSITION",
    "DEFENDING IN ATTACKING THIRD"
]

In [7]:
teams = events_df["Team"].dropna().unique()

team_1, team_2 = teams[:2]

def get_frame_team(row):
    if row["code"] in attacking_codes:
        return row["Team"]

    elif row["code"] in defending_codes:
        return team_2 if row["Team"] == team_1 else team_1

    return None

In [8]:
events_df

,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry,end_frame,start_frame
103,104,1,0 days 00:07:22.760000,0 days 00:08:33.600000,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN,12840,11069
403,404,1,0 days 00:07:23.400000,0 days 00:07:34.760000,LONG BALL,Atletic Club,NaN,1st Half,None,NaN,NaN,11369,11085
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN,11222,11097
542,543,1,0 days 00:07:28.160000,0 days 00:08:00.840000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,1st Half,None,NaN,NaN,12021,11204
318,319,1,0 days 00:07:28.160000,0 days 00:08:00.840000,PROGRESSION,Atletic Club,NaN,1st Half,None,NaN,NaN,12021,11204
...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,397,1,0 days 02:05:10.080000,0 days 02:05:26.440000,LONG BALL,Barcelona,NaN,2nd Half,None,NaN,NaN,188161,187752
448,449,1,0 days 02:05:37.280000,0 days 02:06:12.280000,SET PIECES,N/A,Throw-in,2nd Half,None,NaN,NaN,189307,188432
500,501,1,0 days 02:05:46.640000,0 days 02:06:09.440000,DEFENDING IN MIDDLE THIRD,Barcelona,NaN,2nd Half,None,NaN,NaN,189236,188666
276,277,1,0 days 02:05:46.640000,0 days 02:06:09.440000,PROGRESSION,Atletic Club,NaN,2nd Half,None,NaN,NaN,189236,188666


In [14]:
events_df = events_df[["start_frame", "end_frame", "code"]]
events_df.to_csv("kino_events.csv", index=False)

In [9]:
frame_teams = {}

for _, row in events_df.iterrows():
    team = get_frame_team(row)

    if team is None:
        continue

    for frame in range(
        int(row["start_frame"]),
        int(row["end_frame"]) + 1
    ):
        frame_teams[frame] = team

frame_df = pd.DataFrame(
    frame_teams.items(),
    columns=["frame", "Team"]
).sort_values("frame").reset_index(drop=True)


In [10]:
frame_df.Team.value_counts()

Team
Barcelona       50105
Atletic Club    40662
N/A             18835
Name: count, dtype: int64

In [11]:
frame_df

,frame,Team
0,11069,Atletic Club
1,11070,Atletic Club
2,11071,Atletic Club
3,11072,Atletic Club
4,11073,Atletic Club
...,...,...
109597,189232,Atletic Club
109598,189233,Atletic Club
109599,189234,Atletic Club
109600,189235,Atletic Club


In [12]:
print(sorted_df)

NameError: name 'sorted_df' is not defined

## What is in it

Each row is one tagged interval:

- **timestamp / end_timestamp** — start and end, as pandas timedeltas from the first frame
- **code** — the phase or set piece, upper case (`SET PIECES`, `BUILD UP`, …)
- **Team** — which side the phase belongs to
- **Half** — `1st Half` or `2nd Half`, as strings
- plus per-code tags: `Type`, `Side`, `Max Players in the box`, `Direction of ball entry`

Note the capital in `Team`: kloppy passes the tag groups through with the names the XML uses, so
every tag column is title case. The full vocabulary is in [DATA.md](../DATA.md).

In [ ]:
events['Side'].value_counts()

Side
Central    59
Left       19
Right      15
Name: count, dtype: int64

In [ ]:
events['Direction of ball entry'].value_counts()

Direction of ball entry
Horizontal    34
Vertical      25
Diagonal      13
Name: count, dtype: int64

In [ ]:
events['Max Players in the box'].value_counts()

Max Players in the box
4     65
1     59
2     43
5     31
6     27
3     27
7+    10
Name: count, dtype: int64

In [ ]:
events['Type'].value_counts()

Type
Throw-in                 26
Corner Kick              12
Goal Kick                10
Free Kick                 5
Kick Off (After Goal)     4
Kick Off (Start)          2
Name: count, dtype: int64

In [ ]:
events['Team'].value_counts()

Team
Barcelona       314
Atletic Club    288
N/A              36
Name: count, dtype: int64

### Codes are upper case, tag values are title case

`SET PIECES` and `Kick Off (Start)`. Match case-insensitively unless you enjoy silent empty
results.

In [ ]:
set_pieces = events[events['code'].str.lower() == 'set pieces']
set_pieces['Type'].value_counts()

## Converting to video frames

The timestamps are **seconds from the first frame of the tactical video**, so a video frame —
and therefore a submission row — is just `round(seconds * 25)`. There is no offset to discover.

`starter/load_smart_tagging.py` does this without kloppy, and re-checks the time base against
both kickoffs every time it runs.

In [ ]:
FPS = games.GAMES[games.GAME]['fps']

frames = pd.DataFrame({
    'code': events['code'],
    'Team': events['Team'],
    'start_frame': (events['timestamp'].dt.total_seconds() * FPS).round().astype(int),
    'end_frame': (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int),
})
frames.head()

## Two things to be careful about

### 1. `Team` is not always a team

36 instances are tagged `N/A` — including **every throw-in**, which are exactly the restarts
where knowing the side would be worth the most. Handle it explicitly; do not let it fall
through to a default.

In [ ]:
events['Team'].value_counts(dropna=False)

### 2. An interval is a window, not an instant

A set piece is tagged as a span of a few seconds *around* the restart, not at the frame the ball
is struck. The two `Kick Off (Start)` tags below both **contain** the real kickoff rather than
starting on it — they open about 3.5 seconds early.

So `start_frame` is not the restart frame, and the same imprecision applies to every other
interval in the file. Use a tag to say *roughly when* something happened, and the tracking to say
exactly when.

In [ ]:
kickoffs = set_pieces[set_pieces['Type'].str.lower() == 'kick off (start)'].copy()
kickoffs['start_frame'] = (kickoffs['timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['end_frame'] = (kickoffs['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['real_kickoff'] = [11185, 113742]          # from the ground truth
kickoffs['contains_it'] = ((kickoffs['start_frame'] <= kickoffs['real_kickoff'])
                           & (kickoffs['real_kickoff'] <= kickoffs['end_frame']))

kickoffs[['Half', 'start_frame', 'end_frame', 'real_kickoff', 'contains_it']]

---

## Why this is worth using anyway

Every phase carries a **team** tag, and every possession has a team. Every set piece is a
restart, and every restart ends a dead-ball stretch. The correspondence is not exact — that is
the challenge — but it is a strong prior, for free, on both halves of the label you are asked
to produce.

Just remember the intervals are *phases*, not possessions: a Build Up can span one possession,
stop short of one, or cover two.

**Next:** [`02_tracking_atd.ipynb`](02_tracking_atd.ipynb) for the tracking data, then
[`03_first_submission.ipynb`](03_first_submission.ipynb) to build and score a submission.